In [ ]:
import glob
import os

import numpy as np                  # Numerical operations
import pandas as pd                 # Data manipulation

import matplotlib.pyplot as plt     # Basic plotting
import seaborn as sns               # Statistical data visualization

from scipy import stats             # Statistical functions
from scipy.stats import norm, ttest_ind, chi2_contingency  # Specific tests

import statsmodels.api as sm        # Statistical models
import statsmodels.formula.api as smf  # Formula-based models
from statsmodels.tsa.seasonal import seasonal_decompose  # Seasonal decomposition
from statsmodels.tsa.stattools import adfuller, kpss      # Stationarity tests
from statsmodels.tsa.arima.model import ARIMA             # ARIMA models
from statsmodels.tsa.holtwinters import ExponentialSmoothing  # Exponential smoothing

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score  # Data splitting and tuning
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder          # Data preprocessing
from sklearn.pipeline import Pipeline                                                # ML pipelines
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score   # Classification metrics
from sklearn.metrics import mean_squared_error, r2_score                              # Regression metrics
from sklearn.linear_model import LinearRegression, LogisticRegression                 # Linear models
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor                # Decision trees
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier       # Ensemble methods
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor         # Ensemble methods
from sklearn.svm import SVC, SVR                                                     # Support Vector Machines
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor               # K-Nearest Neighbors
from sklearn.cluster import KMeans, DBSCAN                                           # Clustering
from sklearn.decomposition import PCA, TruncatedSVD                                  # Dimensionality reduction

from sklearn.metrics import confusion_matrix, classification_report         # Model evaluation
from sklearn.metrics import roc_auc_score, roc_curve, auc                   # AUC and ROC
from sklearn.metrics import mean_absolute_error, mean_squared_error         # Regression evaluation


## GDP data

In [ ]:
gdp_data = pd.read_excel('./data/gdp/gdp.xlsx')
gdp_data.iloc[:, 1:] = gdp_data.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
deflator = gdp_data.iloc[-1, 1:]
gdp_data.iloc[:-1, 1:] = gdp_data.iloc[:-1, 1:].div(deflator, axis=1) * 100 * 1000
gdp_data = gdp_data[:-1]
gdp_data

In [ ]:
gdp_data = gdp_data.melt(id_vars=['Регион'], var_name='Год', value_name='ВВП')

In [ ]:
gdp_data

In [ ]:
gdp_data['ВВП_прош'] = gdp_data.groupby("Регион")["ВВП"].shift(1)
gdp_data

## Population data

In [ ]:
population_data = pd.read_excel('./data/population/population.xlsx')
population_data.iloc[:, 1:] = population_data.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
population_data

In [ ]:
population_data = population_data.melt(id_vars=['Регион'], var_name='Год', value_name='Население')

In [ ]:
population_data

## Investment data

In [ ]:
investment_files = glob.glob(os.path.join('./data/investment/', "*.xlsx"))

# Initialize an empty list to store each year's data
all_data = []

# Loop over each file and load it
for file in investment_files:
    # Extract the year from the filename (assuming it’s in the filename, e.g., 'data_2020.xlsx')
    year = file.split('\\')[-1].replace('.xlsx', '')
    
    # Load the Excel file into a DataFrame
    df = pd.read_excel(file)
    
    # Add a 'Year' column
    df['Год'] = int(year)
    
    # Melt the DataFrame to make it long-format: one row per sector, region, and year
    df = df.melt(id_vars=['Год', 'Отрасль'], var_name='Регион', value_name='Инвестиции')
    
    # Append the processed DataFrame to the list
    all_data.append(df)

# Concatenate all years' data into a single DataFrame
investment_data = pd.concat(all_data, ignore_index=True)
investment_data["Отрасль"] = investment_data["Отрасль"].replace({
    "Водоснабжение; водоотведение; сбор, обработка и удаление отходов, деятельность по ликвидации загрязнений": "Водоснабжение; сбор, обработка и удаление отходов, деятельность по ликвидации загрязнений",
})
investment_data['Инвестиции'] = investment_data['Инвестиции'].apply(pd.to_numeric, errors='coerce')
investment_data

In [ ]:
investment_data = investment_data.pivot_table(index=['Год', 'Регион'], columns='Отрасль', values='Инвестиции', fill_value=0)
investment_data.reset_index(inplace=True)
investment_data

In [ ]:
print(investment_data['Отрасль'].unique())
print(len(investment_data['Отрасль'].unique()))

## Merging

In [ ]:
# temp = pd.merge(investment_data, population_data, on=['Год', 'Регион'], how='inner')
merged = pd.merge(investment_data, gdp_data, on=['Год', 'Регион'], how='inner')
merged

## GDP Prediction

In [ ]:
data = pd.get_dummies(merged, columns=['Регион'], drop_first=True)

In [ ]:
X = data.drop(columns=['ВВП'])  # Drop the target and any redundant columns
y = data['ВВП']  # Target variable

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Assuming 'Регион' columns were one-hot encoded, you can use the original 'Регион' column for stratification
stratified_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Since StratifiedShuffleSplit only provides indices, let's use it to split X and y
for train_index, test_index in stratified_split.split(X, merged['Регион']):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

In [ ]:
X_train

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Initialize the model
model = RandomForestRegressor(random_state=42, n_estimators=80)

# Fit the model on the training data
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mse}")